In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: פונקציית Qiskit מאת Global Data Quantum


*ראה את [הפניית ה-API](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan ו-On-Prem (דרך IBM Quantum Platform API) Plan. הן נמצאות בגרסת תצוגה מקדימה וצפויות לשינויים.
## סקירה כללית
ה-Quantum Portfolio Optimizer הוא פונקציית Qiskit המתמודדת עם בעיית אופטימיזציה דינמית של תיק השקעות — בעיה סטנדרטית בתחום הפיננסים, שמטרתה לאזן מחדש השקעות תקופתיות בקבוצת נכסים כדי למקסם תשואות ולמזער סיכונים. הפונקציה מנצלת טכניקות אופטימיזציה קוונטיות מתקדמות ומפשטת את התהליך כך שמשתמשים, ללא ניסיון בחישוב קוונטי, יוכלו ליהנות מיתרונותיה במציאת מסלולי השקעה אופטימליים. הכלי מתאים במיוחד למנהלי תיקים, חוקרים בתחום הפיננסים הכמותיים ומשקיעים פרטיים, ומאפשר בחינה לאחור (back-testing) של אסטרטגיות מסחר באופטימיזציה של תיקי השקעות.
## תיאור הפונקציה
פונקציית ה-Quantum Portfolio Optimizer משתמשת באלגוריתם VQE (Variational Quantum Eigensolver) לפתרון בעיית QUBO (Quadratic Unconstrained Binary Optimization), ובכך מטפלת בבעיות אופטימיזציה דינמית של תיקי השקעות. המשתמשים צריכים רק לספק את נתוני מחיר הנכסים ולהגדיר את אילוץ ההשקעה, ואז הפונקציה מריצה את תהליך האופטימיזציה הקוונטית ומחזירה קבוצה של מסלולי השקעה אופטימליים.

התהליך מורכב מארבעה שלבים עיקריים. ראשית, נתוני הקלט ממופים לבעיה תואמת-קוונטית, תוך בניית ה-QUBO של בעיית אופטימיזציה דינמית של תיק ההשקעות והמרתה לאופרטור קוונטי (Ising Hamiltonian). לאחר מכן, בעיית הקלט ואלגוריתם ה-VQE מותאמים להרצה על חומרה קוונטית. לאחר מכן מופעל אלגוריתם VQE על החומרה הקוונטית, ולבסוף, התוצאות עוברות עיבוד לאחור (post-processing) על מנת לספק את מסלולי ההשקעה האופטימליים. המערכת כוללת גם עיבוד לאחור מודע-רעש (מבוסס [SQD](/guides/qiskit-addons-sqd)) לשיפור איכות הפלט.

פונקציית Qiskit זו מבוססת על [המאמר המדעי שפורסם](https://arxiv.org/abs/2412.19150) מאת Global Data Quantum.
![הדמיה של זרימת העבודה של הפונקציה](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# התחלה
בצע אימות באמצעות [מפתח ה-API](https://quantum.cloud.ibm.com/) שלך ובחר את פונקציית Qiskit כדלקמן. (קטע קוד זה מניח שכבר [שמרת את חשבונך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## דוגמה: אופטימיזציה דינמית של תיק השקעות עם שבעה נכסים
דוגמה זו מדגימה כיצד להריץ את פונקציית אופטימיזציה דינמית של תיק ההשקעות (DPO) ולכוונן את הגדרותיה לביצועים אופטימליים. היא כוללת שלבים מפורטים לכוונון עדין של הפרמטרים להשגת תוצאות רצויות.

מקרה זה כולל שבעה נכסים, ארבעה צעדי זמן וארבעה qubits רזולוציה, מה שמביא לדרישה כוללת של 112 qubits.
### 1. קרא את הנכסים הכלולים בתיק ההשקעות.
אם כל הנכסים בתיק מאוחסנים בתיקייה בנתיב ספציפי, תוכל לטעון אותם ל-`pandas.DataFrame` ולהמיר אותם לאובייקט בפורמט `dict` באמצעות הפונקציה הבאה.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

בדוגמה זו השתמשנו בנכסים [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y), ו-[XS2239553048](https://finance.yahoo.com/quote/XS2239553048). האיור הבא ממחיש את הנתונים שנעשה בהם שימוש בדוגמה זו, ומציג את התפתחות מחיר הסגירה היומי של הנכסים מ-1 בינואר עד 1 בספטמבר 2023.

בדוגמה זו, כדי להבטיח אחידות בין תאריכים, מילאנו ימי לא-מסחר במחיר הסגירה מהתאריך הזמין הקודם. אנחנו מיישמים שלב זה מכיוון שהנכסים שנבחרו מגיעים משווקים שונים עם ימי מסחר משתנים, מה שהופך את תקנון מערך הנתונים לעקביות לחיוני.
![הדמיה של הנתונים ההיסטוריים של הנכסים](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. הגדר את הבעיה.
הגדר את מפרטי הבעיה על ידי קביעת הפרמטרים במילון `qubo_settings`.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. הגדר את הגדרות האופטימייזר וה-ansatz (אופציונלי)
הגדר אופציונלית דרישות ספציפיות לתהליך האופטימיזציה, כולל בחירת האופטימייזר ופרמטריו, וכן ציון ה-primitive ותצורותיו.

עבור ה-Tailored Ansatz, גודל האוכלוסייה שנבחר התבסס על ניסויים קודמים שהראו שערך זה מניב אופטימיזציה יציבה ויעילה.

במקרה של Real Amplitudes Ansatz, ניתן לפעול לפי קשר ליניארי בין ``population_size`` לבין מספר ה-Qubits במעגל. כחוק אצבע משוערת, מומלץ להשתמש ב-**מינימום** ``population_size ~ 0.8 * n_qubits`` עבור ה-ansatz של `real_amplitudes`.

צפוי שה-Optimized Real Amplitudes יניב ביצועי אופטימיזציה טובים יותר מה-ansatz של Real Amplitudes. עם זאת, מספר המשתנים לאופטימיזציה ב-ansatz זה גדל הרבה יותר מהר מאשר במקרה של Real Amplitudes (ראה [המאמר המדעי](https://arxiv.org/pdf/2412.19150)). לפיכך, עבור בעיות גדולות, ה-Optimized Real Amplitudes דורש יותר הרצות מעגלים. ה-Optimized Real Amplitudes ככל הנראה שימושי עבור בעיות הדורשות עד 100 qubits, אך מומלץ לשים לב בעת קביעת פרמטרי ``population_size``. כדוגמה להגדלה זו ב-``population_size``, הטבלה הקודמת מציגה שעבור בעיית 84 qubits, ה-Optimize Real Amplitudes דורש ``population_size`` של 120, בעוד שעבור בעיית 56 qubits, מספיק ``population_size`` של 40.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

ניתן גם לבחור ansatz ספציפי. הדוגמה הבאה משתמשת ב-ansatz של ``'Tailored'``.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. הרץ את הבעיה.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. שלוף תוצאות
הפונקציה מחזירה מילון עם מסלולי ההשקעה מסודרים מהנמוך לגבוה בהתאם לערך פונקציית המטרה שלהם (ראה את החלק [Output](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) בהפניית ה-API). קבוצת תוצאות זו מאפשרת לזהות את המסלול בעל העלות הנמוכה ביותר והערכות ההשקעה המתאימות לו. בנוסף, היא מספקת ניתוח של מסלולים שונים, מה שמקל על בחירת אלה המתאימים ביותר לצרכים או למטרות ספציפיות. גמישות זו מבטיחה שניתן להתאים את הבחירות לסוגים שונים של העדפות או תרחישים.
התחל בהצגת אסטרטגיית התוצאות שהשיגה את עלות המטרה הנמוכה ביותר שנמצאה במהלך התהליך.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


לאחר מכן, באמצעות המטא-נתונים, תוכל לגשת לתוצאות של כל האסטרטגיות שנדגמו. כך תוכל לנתח עוד יותר את המסלולים החלופיים שהאופטימייזר החזיר. לשם כך, קרא את המילון המאוחסן ב-`dpo_result['metadata']['all_samples_metrics']`, המכיל לא רק מידע נוסף על האסטרטגיה האופטימלית, אלא גם פרטים על אסטרטגיות המועמדות האחרות שהוערכו במהלך האופטימיזציה.

הדוגמה הבאה מציגה כיצד לקרוא מידע זה באמצעות `pandas` כדי לחלץ מדדי מפתח הקשורים לאסטרטגיה האופטימלית. אלה כוללים סטיית אילוץ, יחס Sharpe והתשואה המקבילה על ההשקעה.